In [8]:
import pandas as pd
import difflib

df = pd.read_csv('planilha.csv', sep=None, engine='python');

colunas_para_excluir = ['Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14'];
df = df.drop(columns=colunas_para_excluir);

df = df.fillna(0);

df['CLIENTE'] = df['CLIENTE'].str.upper();
df['SERVIÇO'] = df['SERVIÇO'].str.upper();
df['FORMA PGTO'] = df['FORMA PGTO'].str.upper();
df['ANIMAL'] = df['ANIMAL'].str.upper();

df['DATA'] = pd.to_datetime(df['DATA'], dayfirst=True, errors='coerce')

# Criar uma nova coluna só com ano e mês
df['ANO_MES'] = df['DATA'].dt.to_period('M')

# Definir o intervalo de filtro em período mensal
inicio = pd.Period("2024-04", freq='M')
fim = pd.Period("2025-05", freq='M')

# Aplicar o filtro usando a nova coluna
df = df.loc[(df['ANO_MES'] >= inicio) & (df['ANO_MES'] <= fim)]

def categorizar_servico(servico):
    servico = servico.strip()  # remove espaços antes/depois

    if 'PACOTE' in servico:
        return 'PACOTE'
    if 'BANHO' in servico and 'TOSA' in servico:
        return 'BANHO+TOSA'
    if 'BANHO' in servico:
        return 'BANHO'
    if 'TOSA' in servico or 'DIFER TOSA' in servico:
        return 'TOSA'
    if 'TRANSP' in servico:
        return 'TRANSPORTE'
    if 'HIDRAT' in servico:
        return 'HIDRATAÇÃO'
    if 'DESEMB' in servico:
        return 'DESEMBARAÇO'
    if 'CORTE' in servico and 'UNHA' in servico:
        return 'CORTE DE UNHA'
    if 'CLINICA' in servico:
        return 'CLÍNICA'

    return servico  # se nada bater, retorna o original

# Aplica categorização antes da contagem
df['SERVIÇO'] = df['SERVIÇO'].apply(categorizar_servico)

contagem_servicos = df['SERVIÇO'].value_counts()

servicos_base = contagem_servicos[contagem_servicos >= 5].index.tolist()

mapeamento = {}
for servico in contagem_servicos[contagem_servicos < 5].index:
    similares = difflib.get_close_matches(servico, servicos_base, n=1, cutoff=0.02)
    if similares:
        mapeamento[servico] = similares[0]

df['SERVIÇO'] = df['SERVIÇO'].replace(mapeamento)

df = df.drop(columns = ' ID');
df = df.dropna();
display(df.head());

,CLIENTE,ANIMAL,DATA,SERVIÇO,FORMA PGTO,VALOR,MAQ CL.,MAQ BAN.,ANO_MES
1,THAISA FONTANA,PIPOCA,2024-04-01,BANHO+TOSA,DINHEIRO,"95,00","95,00",0,2024-04
2,GIAN CRISTIANO,RUBI,2024-04-01,BANHO,DINHEIRO,"70,00","70,00",0,2024-04
3,TATIANE FERREIRA,BENEDITA,2024-04-01,BANHO+TOSA,CREDITO,"110,00",0,110,2024-04
4,HELEN HAIR,CHECHE,2024-04-01,BANHO+TOSA,PIX SANTANDER,"190,00","190,00",0,2024-04
5,VANESSA MENDES,MEL,2024-04-01,PACOTE,DEBITO,"200,00","200,00",0,2024-04


In [9]:
import pandas as pd

dataframe = pd.read_csv('relatorio.csv', sep=None, engine='python');

df['DATA'] = pd.to_datetime(df['DATA'], dayfirst=True, errors='coerce')

dataframe = dataframe.dropna();
display(dataframe.head());

,CLIENTE,ANIMAL,DATA,SERVIÇO,FORMA PGTO,VALOR,MAQ CL.,MAQ BAN.,SERVIÇO_LIMPO,CATEGORIA
0,THAISA FONTANA,PIPOCA,04/24,BANHO+TOSA,DINHEIRO,"95,00","95,00",0,BANHO+TOSA,BANHO + TOSA
1,GIAN CRISTIANO,RUBI,04/24,BANHO,DINHEIRO,"70,00","70,00",0,BANHO,BANHO
2,TATIANE FERREIRA,BENEDITA,04/24,BANHO+TOSA,CREDITO,"110,00",0,110,BANHO+TOSA,BANHO + TOSA
3,HELEN HAIR,CHECHE,04/24,BANHO+TOSA+TRANSP,PIX SANTANDER,"190,00","190,00",0,BANHO+TOSA+TRANSP,BANHO + TOSA + TRANSPORTE
4,VANESSA MENDES,MEL,04/24,PACOTE,DEBITO,"200,00","200,00",0,PACOTE,PACOTE


In [10]:
import plotly.express as px

# for coluna in df.columns:
#     grafico = px.histogram(df, x=coluna, color="SERVIÇO", text_auto=True)
#     grafico.show();
df["ANO_MES"] = df["ANO_MES"].astype(str)
grafico = px.histogram(df, x="ANO_MES", color="SERVIÇO", text_auto=True)
grafico.show();
grafico.write_html("grafico_interativo.html")
df.to_csv("dados_filtrados.csv", index=False, encoding='utf-8-sig')

In [11]:

print("Todos os serviços únicos encontrados:")
servicos_completos = df['SERVIÇO'].value_counts()

# Mostrar todos os valores
pd.set_option('display.max_rows', None)  # Remove limite de linhas
print(servicos_completos)

# Ou salvar em um arquivo para análise
servicos_completos.to_csv('servicos_unicos.csv')
print("\nArquivo 'servicos_unicos.csv' criado com todos os serviços!")

Todos os serviços únicos encontrados:
SERVIÇO
PACOTE           1557
BANHO+TOSA        550
BANHO             501
TOSA              369
CLÍNICA            43
TRANSPORTE         43
DESEMBARAÇO        22
CORTE DE UNHA      19
Name: count, dtype: int64

Arquivo 'servicos_unicos.csv' criado com todos os serviços!


In [6]:
import plotly.express as px

# for coluna in df.columns:
#     grafico = px.histogram(df, x=coluna, color="SERVIÇO", text_auto=True)
#     grafico.show();

grafico = px.histogram(dataframe, x="DATA", color="SERVIÇO", text_auto=True)
grafico.show();
grafico.write_html("grafico_interativo.html")